In [12]:
import pandas as pd
import numpy as np
import os
from PIL import Image
from scipy.signal import stft
from tqdm import tqdm

# 데이터 불러오기
signal_df = pd.read_csv('X_train.csv')
label_df = pd.read_csv('y_train.csv')

save_dir = 'stft_images_224'
os.makedirs(save_dir, exist_ok=True)

def create_stft_image(signal_row, save_path):
    signal = np.array(signal_row)

    f, t, Zxx = stft(signal, nperseg=min(32, len(signal)))
    spectrogram = np.abs(Zxx)

    # 정규화 후 이미지 변환
    spectrogram = (spectrogram - np.min(spectrogram)) / (np.max(spectrogram) - np.min(spectrogram) + 1e-8)
    spectrogram = (spectrogram * 255).astype(np.uint8)
    img = Image.fromarray(spectrogram)
    img = img.convert("RGB")
    img = img.resize((224, 224))
    img.save(save_path)

image_paths = []
labels = []

for idx, (signal_row, label_row) in tqdm(enumerate(zip(signal_df.values, label_df.values)), total=len(signal_df)):
    save_path = os.path.join(save_dir, f"{idx}.png")
    create_stft_image(signal_row, save_path)
    image_paths.append(save_path)
    labels.append(np.argmax(label_row))

result_df = pd.DataFrame({
    'image_path': image_paths,
    'label': labels
})
result_df.to_csv('stft_image_label_mapping_224.csv', index=False)

print("✅ 224x224 STFT 이미지 변환 완료")


100%|███████████████████████████████████████████████████████████████████████████| 18805/18805 [01:36<00:00, 194.73it/s]


✅ 224x224 STFT 이미지 변환 완료
